In [4]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv 
load_dotenv() 

os.environ["Google_API_KEY"] = os.getenv("Google_API_Key")

model=init_chat_model("google_genai:gemini-3.1-flash-lite-preview")
response=model.invoke("What is the capital of India")
response

AIMessage(content=[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'EjQKMgEMOdbHYWSqhu8LIUz1Hfmqql11tMBtv8pTzSSmzL8wmAFNmKZfPNbVXePwfg+ibm5C'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019df151-cef1-7813-bedf-079f280f9552-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 9, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})

In [17]:
from langchain.tools import tool 

@tool
def get_weather(location: str) -> str:
    """Get the weather of the given location."""
    return f"The weather in {location} is sunny."

model_with_tool = model.bind_tools([get_weather])

response=model_with_tool.invoke("What's the weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'589491e4-dc66-4d52-96b2-5f87f69f4a4b': 'EjQKMgEMOdbHsDLD9VCP+50BrCSqMW3SjhuQXzA7JchiovOUiDymlL/Z5yMMuxLDEHfRTxwL'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019df162-dd48-7ad1-8b7a-d39a0d3601c0-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': '589491e4-dc66-4d52-96b2-5f87f69f4a4b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 55, 'output_tokens': 17, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}}
Tool: get_weather
Args: {'location': 'New York'}


In [19]:
#user question → model calls tool → tool executes → result goes back to model → final answer

messages = [{'role':'user', "content":"What is the weather in New York?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)


for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tool.invoke(messages)
print(final_response)

content=[{'type': 'text', 'text': 'The weather in New York is currently sunny.', 'extras': {'signature': 'EjQKMgEMOdbHpVwFZRhRSjbeRRUqw+1qkWxPHR7Wn0aILZOAeUl+0RKSgd31v49u6KOYqHfX'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019df166-2a25-7ca1-be70-3f8fb6709982-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 92, 'output_tokens': 9, 'total_tokens': 101, 'input_token_details': {'cache_read': 0}}
